# Appendix B | Linear Regression Model

---

# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.read_csv_to_df import read_csv_to_df
from utils.rule_of_thumb__skewness import rule_of_thumb__skewness

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer # IterativeImputer
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold

# Defaults

In [2]:
SEED = 42
np.random.seed(SEED)

TARGET = 'Life Ladder'

# Fetch Data

In [3]:
train_path = '/Q1_OLD/DATA/happiness_data_train.csv'
train_df = read_csv_to_df(path=train_path)

X = train_df.drop(columns=TARGET)
y = train_df[TARGET]

print(f'X: {X.shape[0]} rows x {X.shape[1]} features, y: {TARGET}')

X: 1559 rows x 8 features, y: Life Ladder


# Base Experiment 1 _(testing step options)_

In [4]:
pipe = Pipeline([
	('impute', SimpleImputer()),
	('transform', 'passthrough'),
	('scale', StandardScaler()),
	('model', LinearRegression()),
])

In [5]:
# Every option you want to try for each step
param_grid = {
	'impute': [
		SimpleImputer(strategy='mean'),
		SimpleImputer(strategy='median'),
		KNNImputer(),
		# IterativeImputer(random_state=SEED)
	],
	'transform': [
		'passthrough',
		PowerTransformer(standardize=False),
		QuantileTransformer(output_distribution='normal')
	],
	'scale': [
		StandardScaler(), MinMaxScaler()
	],
}

In [6]:
search = GridSearchCV(pipe, param_grid, cv=KFold(4, shuffle=True, random_state=SEED),
                      scoring='neg_root_mean_squared_error', n_jobs=-1)
search.fit(X, y)

results = pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_impute,param_scale,param_transform,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
13,0.015476,0.001613,0.003746,0.000464,KNNImputer(),StandardScaler(),PowerTransformer(standardize=False),"{'impute': KNNImputer(), 'scale': StandardScal...",-0.571844,-0.514185,-0.509515,-0.508252,-0.525949,0.026590,1
16,0.015048,0.002445,0.003285,0.000129,KNNImputer(),MinMaxScaler(),PowerTransformer(standardize=False),"{'impute': KNNImputer(), 'scale': MinMaxScaler...",-0.571844,-0.514185,-0.509515,-0.508252,-0.525949,0.026590,1
10,0.009571,0.000844,0.000871,0.000056,SimpleImputer(strategy='median'),MinMaxScaler(),PowerTransformer(standardize=False),"{'impute': SimpleImputer(strategy='median'), '...",-0.575395,-0.509258,-0.510829,-0.509875,-0.526339,0.028328,3
7,0.009607,0.001150,0.000848,0.000034,SimpleImputer(strategy='median'),StandardScaler(),PowerTransformer(standardize=False),"{'impute': SimpleImputer(strategy='median'), '...",-0.575395,-0.509258,-0.510829,-0.509875,-0.526339,0.028328,4
1,0.013731,0.005060,0.001162,0.000444,SimpleImputer(),StandardScaler(),PowerTransformer(standardize=False),"{'impute': SimpleImputer(), 'scale': StandardS...",-0.575721,-0.509831,-0.511017,-0.510059,-0.526657,0.028331,5
4,0.011438,0.004175,0.001208,0.000626,SimpleImputer(),MinMaxScaler(),PowerTransformer(standardize=False),"{'impute': SimpleImputer(), 'scale': MinMaxSca...",-0.575721,-0.509831,-0.511017,-0.510059,-0.526657,0.028331,5
15,0.008595,0.001872,0.003087,0.000026,KNNImputer(),MinMaxScaler(),passthrough,"{'impute': KNNImputer(), 'scale': MinMaxScaler...",-0.597582,-0.531240,-0.532936,-0.528742,-0.547625,0.028881,7
12,0.009162,0.002260,0.003110,0.000060,KNNImputer(),StandardScaler(),passthrough,"{'impute': KNNImputer(), 'scale': StandardScal...",-0.597582,-0.531240,-0.532936,-0.528742,-0.547625,0.028881,7
14,0.019299,0.001675,0.004863,0.000458,KNNImputer(),StandardScaler(),QuantileTransformer(output_distribution='normal'),"{'impute': KNNImputer(), 'scale': StandardScal...",-0.590551,-0.535973,-0.534273,-0.529720,-0.547629,0.024886,9
17,0.015110,0.001853,0.004399,0.000085,KNNImputer(),MinMaxScaler(),QuantileTransformer(output_distribution='normal'),"{'impute': KNNImputer(), 'scale': MinMaxScaler...",-0.590551,-0.535973,-0.534273,-0.529720,-0.547629,0.024886,10


# Base Experiment 2 _(testing pipe ordering & step options)_

In [7]:
pipe = Pipeline([
      ('impute_first', 'passthrough'),
      ('transform', 'passthrough'),
      ('scale', 'passthrough'),
      ('impute_last', 'passthrough'),
      ('model', LinearRegression()),
])

In [8]:
IMPUTERS = [
	SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer(),
	# IterativeImputer(random_state=SEED)
]
TRANSFORMS = ['passthrough', PowerTransformer(standardize=False), QuantileTransformer(output_distribution='normal')]
SCALERS = [StandardScaler(), MinMaxScaler()]

In [9]:
# A list of grids: each dict is searched separately, so only one imputer slot is ever filled
param_grid = [
      # impute -> transform -> scale -> model
      {'impute_first': IMPUTERS, 'transform': TRANSFORMS, 'scale': SCALERS, 'impute_last': ['passthrough']},
      # transform -> scale -> impute -> model
      {'impute_first': ['passthrough'], 'transform': TRANSFORMS, 'scale': SCALERS, 'impute_last': IMPUTERS},
]

In [10]:
search = GridSearchCV(pipe, param_grid, cv=KFold(4, shuffle=True, random_state=SEED),
                      scoring='neg_root_mean_squared_error', n_jobs=-1)
search.fit(X, y)

GridSearchCV(cv=KFold(n_splits=4, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('impute_first', 'passthrough'),
                                       ('transform', 'passthrough'),
                                       ('scale', 'passthrough'),
                                       ('impute_last', 'passthrough'),
                                       ('model', LinearRegression())]),
             n_jobs=-1,
             param_grid=[{'impute_first': [SimpleImputer(),
                                           SimpleImputer(strategy='median'),
                                           KNNImputer()],
                          'impute_last': [...
                                        QuantileTransformer(output_distribution='normal')]},
                         {'impute_first': ['passthrough'],
                          'impute_last': [SimpleImputer(),
                                          SimpleImputer(strategy='median'),
                                          KNNImputer()],
                          'scale': [StandardScaler(), MinMaxScaler()],
                          'transform': ['passthrough',
                                        PowerTransformer(standardize=False),
                                        QuantileTransformer(output_distribution='normal')]}],
             scoring='neg_root_mean_squared_error')

In [11]:
results = pd.DataFrame(search.cv_results_)
results['order'] = results['param_impute_first'].map(
      lambda v: 'transform -> scale -> impute' if v == 'passthrough' else 'impute -> transform -> scale')
results['imputer'] = [p['impute_last'] if p['impute_first'] == 'passthrough' else p['impute_first'] for p in results['params']]
results['val RMSE'] = -results['mean_test_score']
results.sort_values('val RMSE')[['order', 'imputer', 'param_transform', 'param_scale', 'val RMSE']]

,order,imputer,param_transform,param_scale,val RMSE
16,impute -> transform -> scale,KNNImputer(),PowerTransformer(standardize=False),MinMaxScaler(),0.525949
13,impute -> transform -> scale,KNNImputer(),PowerTransformer(standardize=False),StandardScaler(),0.525949
10,impute -> transform -> scale,SimpleImputer(strategy='median'),PowerTransformer(standardize=False),MinMaxScaler(),0.526339
7,impute -> transform -> scale,SimpleImputer(strategy='median'),PowerTransformer(standardize=False),StandardScaler(),0.526339
22,transform -> scale -> impute,SimpleImputer(),PowerTransformer(standardize=False),MinMaxScaler(),0.526494
19,transform -> scale -> impute,SimpleImputer(),PowerTransformer(standardize=False),StandardScaler(),0.526494
28,transform -> scale -> impute,SimpleImputer(strategy='median'),PowerTransformer(standardize=False),MinMaxScaler(),0.526506
25,transform -> scale -> impute,SimpleImputer(strategy='median'),PowerTransformer(standardize=False),StandardScaler(),0.526506
1,impute -> transform -> scale,SimpleImputer(),PowerTransformer(standardize=False),StandardScaler(),0.526657
4,impute -> transform -> scale,SimpleImputer(),PowerTransformer(standardize=False),MinMaxScaler(),0.526657
